# Reproducing Ray, Moaddab & McDannald (2022) from CRCNS nac-1

**Paper.** Ray MH, Moaddab M, McDannald MA (2022). Threat and bidirectional valence signaling in
the nucleus accumbens core. *J Neurosci* 42(5):817-833.
[doi:10.1523/JNEUROSCI.1107-21.2021](https://doi.org/10.1523/JNEUROSCI.1107-21.2021). The claims
file was written from the version of record (Introduction, Methods, Results, legends and
Discussion); the Extended Data figures were not available.

**Data set.** CRCNS nac-1, [doi:10.6080/K0M906VB](https://doi.org/10.6080/K0M906VB), read from the
public bucket at `https://crcnsarchive.s3-us-west-2.amazonaws.com/nac-1/live/data.zip`, streamed
with range requests. CRCNS data sets are not versioned, so the version is recorded as
`unversioned`, and the file read is pinned by its MD5 in the data set's `checksums.md5`:
`data.zip` ab13407080360d7d683ef818ae66f33a (33,243,283 bytes, last modified 2026-02-11). There is
no other version to choose from: the bucket holds only `live/` for nac-1 (no `old/`), and the
data description is version 0.6 (Feb 9, 2022), its initial version.

**What the data set holds.** One MATLAB file per unit for the 368 units (spike times, mean
waveform, and the session's event channels: cues, shocks, pellets, nose pokes), a precomputed
`cube.mat` that the authors' script builds from those files, and the authors' MATLAB scripts
(`cubeScriptPR25.m` and `NAccAnalysis/*.m`). It holds no pre-surgery behavior sessions and no
population (cluster) labels.

**The authors' scripts.** The paper links no analysis code (its lab page returns 404). The
scripts in the data set are the authors' code, and this notebook takes from them the
details the Methods and the claims file leave open, citing them. They give the event channel
map, which the data description lacks.

Every random draw uses a generator seeded for its own claim section.

| Claim | Data | Status |
|---|---|---|
| units-recorded | data set | tested |
| cue-responsive-193 | data set | tested |
| fig1c-acquisition | data set | blocked: the pre-surgery sessions are not in the data set |
| fig1g-discrimination, fig1h-poke-rate-unrelated, fig1h-discrimination-safety | data set | tested |
| fig2-five-clusters, populations-by-rat, most-units-threat | data set | tested |
| fig3ab-rate-cv-unimodal, fig3c-waveform-bimodal | data set | tested |
| fig4ab, fig4b, fig4cd, fig4ef, fig4f, fig5ab, fig5cd (cue firing by population) | data set | tested |
| fig6-similarity-anova, fig6d, fig6e, fig6f, fig6-thi-greatest-similarity | data set | tested |
| fig7-tuning-curves, fig8ab, fig8cd, fig8ef | data set | tested |
| fig9a, fig9bcd, fig9efg, fig9hij (reward) | data set | tested |
| fig10a, fig10bcd, fig10efg (nose poke cessation) | data set | tested |

In [ ]:
import collections
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats

import nac

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
U = nac.load_units()
CUBE = nac.load_cube()
print(f"loaded {len(U)} unit files from {nac.URL} in {time.time() - started:.0f} s")

## Data check

The loader assigns the cue channels to danger, uncertainty and safety by the program letter in
the file name, as `cubeScriptPR25.m` does, and splits uncertainty trials into shock and omission
by a shock within 30 s of onset. The first check compares what the loader computes from the raw
files with the authors' `cube.mat`: trial onsets and types, the binned firing, and the
suppression ratios.

In [ ]:
cube_names = ["".join(CUBE.name[0][j][i] for j in range(22)) for i in range(len(CUBE.name[0][0]))]
cube_index = {n: i for i, n in enumerate(cube_names)}
raw250 = CUBE.fire[0, 0].raw[0, 0].ms250[0, 0].interval
cer = CUBE.cer[0, 0].cue
n_tag = n_raw = n_sr = 0
for u in U:
    i = cube_index[u["filename"] + u["neuron"]]
    n_tag += np.allclose(CUBE.tag[:, 1, i], u["onsets"]) and np.array_equal(CUBE.tag[:, 0, i], nac.TRIAL_TYPES + 1)
    n_raw += np.array_equal(np.floor(nac.binned_rates(u["spikes"], u["onsets"])), raw250[:, :, i].astype(float))
    n_sr += np.allclose(nac.suppression(u, "one")[0], cer[:, 0, i])
print(f"units in cube.mat: {len(cube_names)}; matched by name: {sum(u['filename'] + u['neuron'] in cube_index for u in U)}")
print(f"trial onsets and types equal to cube.tag: {n_tag} of {len(U)}")
print(f"floored 250-ms rates equal to cube.fire.raw.ms250: {n_raw} of {len(U)}")
print(f"suppression ratios (0/0 set to 1, as the script does) equal to cube.cer.cue: {n_sr} of {len(U)}")
print()
print("trials per type (danger, unc shock, unc omission, safety):", collections.Counter(u["n_type"] for u in U))
print("event channels:", collections.Counter(tuple(u["channels"]) for u in U))
print("programs:", dict(collections.Counter(u["program"] for u in U)))

In [ ]:
units = pd.DataFrame([dict(rat=u["rat"], session=u["session"], program=u["program"], n_spikes=len(u["spikes"]),
                           duration=u["spikes"][-1] - u["spikes"][0], rate=len(u["spikes"]) / (u["spikes"][-1] - u["spikes"][0]),
                           first_onset=u["onsets"].min(), last_onset=u["onsets"].max(), n_poke=len(u["poke"]),
                           n_pellet=len(u["pellet"]), n_shock=len(u["shock"])) for u in U])
print("units, sessions (rat and date) and programs per rat:")
print(units.groupby("rat").agg(units=("session", "size"), sessions=("session", "nunique"),
                               programs=("program", lambda x: "".join(sorted(set(x))))))
print(f"total: {len(units)} units, {units.session.nunique()} sessions, {units.rat.nunique()} rats")
print()
print(units[["n_spikes", "duration", "rate", "n_poke", "n_pellet", "n_shock"]].describe().round(2))
iti = np.concatenate([np.diff(np.sort(u["onsets"])) for u in {u["session"]: u for u in U}.values()])
print(f"\ninter-trial onset intervals: median {np.median(iti):.0f} s, range {iti.min():.0f} to {iti.max():.0f} s (paper: mean ITI 3.5 min)")
sh = []
for u in {u["session"]: u for u in U}.values():
    for t in u["onsets"][nac.TRIAL_TYPES <= 1]:
        d = u["shock"][(u["shock"] > t)] - t
        sh.append(d.min() if len(d) else np.nan)
sh = np.array(sh)
print(f"shock after onset on danger and uncertainty-shock trials: {np.nanmin(sh):.2f} to {np.nanmax(sh):.2f} s (paper: 2 s after the 10-s cue, so 12 s)")
print(f"shocks per session: {collections.Counter(len(u['shock']) for u in {u['session']: u for u in U}.values())} (design: 6; EVT01)")
pb = CUBE.poke[0, 0].base
m20 = m2 = 0
for u in U:
    i = cube_index[u["filename"] + u["neuron"]]
    m20 += np.array_equal(nac.count_in(u["poke"], u["onsets"] - 20, u["onsets"]) * 3, pb[:, 0, i].astype(int))
    m2 += np.array_equal(nac.count_in(u["poke"], u["onsets"] - 2, u["onsets"]) * 30, pb[:, 0, i].astype(int))
print(f"cube.poke.base equals the 20-s pre-cue poke rate (pokes/min) in {m20} units, the 2-s rate in {m2} (description: '2 s prior')")
w = np.array([u["wave"] for u in U])
print(f"waveforms: {w.shape[1]} points (description: 32 points over 0.8 ms), values {w.min():.2e} to {w.max():.2e} (units not given)")

What the checks show. The loader reproduces the authors' cube exactly (trial onsets and types,
binned firing, suppression ratios) for all 368 units, so the event channel map taken from the
script is the one the authors used. Every session has 4, 2, 6 and 4 trials of the four types
and 6 shocks, 12 s after onset, as the Methods describe. Onsets are 203 to 265 s apart
(median 233 s), consistent with a mean ITI of 3.5 min plus the trial. The data description says
the cube's `poke.base` is the rate 2 s before the cue; the values are the 20-s rate (a
description error, recorded as a finding).

Coverage. The files hold spikes, pokes and pellets only from 20 s before each onset to about
33 s after it: the recording is paused between trials. The data description does not say so.

Duplicates and clocks. A recording deposited twice would show identical spike trains; units of
one session must share one set of event times. The clock check uses contrasts that only hold
if the event times and the spikes and pokes share a clock: the suppression of poking to danger
against safety, and the fraction of units whose firing changes at cue onset, each at the true
onsets and at onsets shifted by 60 s (inside the ITI).

In [ ]:
sig = collections.defaultdict(list)
for u in U:
    sig[(len(u["spikes"]), round(float(u["spikes"].sum()), 4))].append(u["file"])
print("units with identical spike trains:", [v for v in sig.values() if len(v) > 1])
by_session = collections.defaultdict(list)
for u in U:
    by_session[u["session"]].append(u)
print("sessions whose units disagree on event times:",
      [s for s, v in by_session.items() if len({(round(float(x["onsets"].sum()), 4), len(x["poke"]), len(x["pellet"])) for x in v}) > 1])
ev_sets = collections.defaultdict(set)
for s, v in by_session.items():
    ev_sets[(round(float(v[0]["onsets"].sum()), 4), len(v[0]["poke"]))].add(s)
print("event sets shared by two sessions:", [x for x in ev_sets.values() if len(x) > 1])
same_name = collections.Counter((u["session"], u["neuron"]) for u in U)
print("session and unit name used twice:", [k for k, c in same_name.items() if c > 1])

cover = []
for u in U:
    on = np.sort(u["onsets"])
    for name in ("spikes", "poke", "pellet"):
        x = u[name]
        if len(x):
            rel = x - on[np.clip(np.searchsorted(on, x + 20, side="right") - 1, 0, 15)]
            cover.append((name, rel.min(), rel.max()))
cover = pd.DataFrame(cover, columns=["stream", "min_rel", "max_rel"]).groupby("stream").agg(["min", "max"])
print("time of every spike, poke and pellet relative to the trial onset before it (s):")
print(cover.round(2))
gaps = np.concatenate([np.diff(u["spikes"]) for u in U])
print(f"inter-spike intervals longer than 100 s: {np.sum(gaps > 100)} (the recording is paused between trials, EVT14)")

def shifted(u, dt):
    v = dict(u)
    v["onsets"] = u["onsets"] + dt
    return v

rows = []
for dt in (0, 60):
    sr = []
    for v in by_session.values():
        s, _ = nac.suppression(shifted(v[0], dt), "drop")
        sr.append([np.nanmean(s[nac.CUE_OF_TRIAL == 0]), np.nanmean(s[nac.CUE_OF_TRIAL == 2])])
    sr = np.array(sr)
    resp = np.mean([nac.screen(shifted(u, dt), "pooled")[0] for u in U])
    rows.append(dict(shift_s=dt, danger_SR=np.nanmean(sr[:, 0]), safety_SR=np.nanmean(sr[:, 1]),
                     sessions_danger_gt_safety=np.mean(sr[:, 0] > sr[:, 1]), units_with_cue_response=resp))
print(pd.DataFrame(rows).round(3).to_string(index=False))

## units-recorded

> 368 NAcc single units were recorded from seven rats over 95 fear discrimination sessions.

A session is a rat and a date (the file name `acRRMMDDYY...`); each date has one behavior
program per rat. The authors' `mainAnalysis.m` counts sessions as distinct rows of
suppression ratios, which is shown as a variant.

In [ ]:
n_units = len(U)
n_sessions = len({u["session"] for u in U})
n_rats = len({u["rat"] for u in U})
n_files = len({u["filename"] for u in U})
sr_rows = len({tuple(np.round(nac.suppression(u, "one")[0], 8)) for u in U})
print(f"units {n_units} (paper 368), sessions {n_sessions} (paper 95), rats {n_rats} (paper 7)")
print(f"distinct rat, date and program: {n_files}; distinct rows of 16 suppression ratios (mainAnalysis.m's count): {sr_rows}")
print("sessions per rat:", dict(sorted(collections.Counter(s[:4] for s in {u["session"] for u in U}).items())))

**Result: partially reproduced.** The data set holds 368 units from 7 rats, as the paper says,
but from 90 distinct sessions (rat and date), not 95; counting distinct rows of suppression
ratios, as the authors' script does, gives 84. The tolerance is zero, so the session count does
not match.

## cue-responsive-193

> The cue-responsive screen identified 193 units (about 53% of all units) from 82 sessions, with
> at least seven cue-responsive units in each rat.

Readings of the criterion: (a) the eight uncertainty trials tested as one cue; (b) shock and
omission trials as two cues. Rates in Hz over the 10-s baseline and the first 1 s and last 5 s
of each cue, paired two-tailed t-tests at p < 0.05. The authors' `cubeScriptPR25.m` floors each
rate to an integer before `mainAnalysis.m` runs the same paired tests (`ttest`, pooled
uncertainty rows 5 to 12); that is shown as a variant. A test with no variance in the
differences gives no p-value and does not count, as MATLAB's `ttest` returns NaN.

In [ ]:
screens = {}
for label, unc, fl in (("(a) uncertainty pooled", "pooled", False), ("(b) shock and omission split", "split", False),
                       ("script: pooled, floored rates", "pooled", True), ("script floor, split", "split", True)):
    screens[label] = np.array([nac.screen(u, unc, fl)[0] for u in U])
rows = []
for label, r in screens.items():
    per_rat = collections.Counter(u["rat"] for u, x in zip(U, r) if x)
    rows.append(dict(reading=label, responsive=int(r.sum()), share=r.mean(), sessions=len({u["session"] for u, x in zip(U, r) if x}),
                     min_per_rat=min(per_rat.values()), per_rat=dict(sorted(per_rat.items())),
                     within_tolerance=abs(r.sum() - 193) <= 19.3 and abs(len({u["session"] for u, x in zip(U, r) if x}) - 82) <= 8.2 and min(per_rat.values()) >= 7))
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("expected: 193 +- 19.3 units, 82 +- 8.2 sessions, at least 7 per rat")
fp = []
rng_screen = np.random.default_rng(101)
for u in U[:368:4]:
    fp.append(nac.screen(shifted(u, 60), "pooled")[0])
print(f"null reference: with onsets shifted by 60 s, {np.mean(fp):.3f} of {len(fp)} units pass reading (a)")

**Result: reproduced.** Under reading (b) the screen finds 211 units from 81 sessions, with at
least 7 per rat, all within tolerance; under reading (a) it finds 219 units, above the
tolerance of 193 +- 19.3, from 82 sessions. The authors' own procedure (floored rates, pooled
uncertainty) finds exactly 193 units from 82 sessions, with at least 7 per rat.

The later claims need the paper's 193 units. They use the set of the authors' procedure, which
gives the paper's count and sessions exactly. This is an assumption: the Methods say "mean
firing rate (Hz)" and do not mention flooring.

In [ ]:
responsive = screens["script: pooled, floored rates"]
R = [u for u, r in zip(U, responsive) if r]
print(f"cue-responsive units used below: {len(R)} from {len({u['session'] for u in R})} sessions")

## fig1c-acquisition

> Over the eight pre-surgery sessions rats acquired discrimination ...

The data set's files are listed below: there is one file per recorded unit, from the
recording sessions only. The pre-surgery sessions are not in it. The authors'
`preRecordingBehavior.m` reads them from a folder (`AC Group Data`) that is not part of the
data set.

In [ ]:
z = nac.open_zip()
folders = collections.Counter(n.rsplit("/", 1)[0] for n in z.namelist() if not n.endswith("/"))
print("files per folder in data.zip:", dict(folders))
print("non-unit files:", [n for n in z.namelist() if not n.startswith("data/matFiles/") and not n.endswith("/")])
dates = sorted({u["date"] for u in U})
print(f"session dates: {len(dates)} distinct, from {dates[0]} to {dates[-1]} (MMDDYY)")

**Result: blocked (data missing).** No file holds the eight pre-surgery sessions.

## Behavior per rat (fig1g, fig1h)

Per session, the mean suppression ratio of each cue over its trials (uncertainty pooled), and
the mean baseline poke rate (pokes per min, 20 s before onset). Per rat, the mean over the
chosen sessions (the authors' `mainAnalysis.m` averages the session rows of each rat). Session
readings of fig1g: (a) the 82 sessions with a cue-responsive unit; (b) all sessions (90 here);
(c) the sessions of the three main populations (computed after the clustering, below). Readings
of a 0/0 suppression ratio from the claims file: left out, or 0; the authors' scripts set it to
1 (variant).

In [ ]:
def behavior(sessions, nan_reading):
    per_rat = collections.defaultdict(list)
    for s in sorted(sessions):
        u = by_session[s][0]
        sr, base = nac.suppression(u, nan_reading)
        per_rat[u["rat"]].append([np.nanmean(sr[nac.CUE_OF_TRIAL == k]) for k in range(3)] + [base.mean()])
    rats = sorted(per_rat)
    return pd.DataFrame([np.mean(per_rat[r], 0) for r in rats], index=rats,
                        columns=["danger", "uncertainty", "safety", "baseline_pokes_per_min"])

def fig1g(beh, rng, mirrored=False):
    F = nac.rm_anova1(beh[["danger", "uncertainty", "safety"]].to_numpy())
    d = {"D-U": beh.danger - beh.uncertainty, "U-S": beh.uncertainty - beh.safety, "D-S": beh.danger - beh.safety}
    out = dict(F=F[0], df=f"{F[1]},{F[2]}", p=F[3])
    for k, x in d.items():
        m, lo, hi = nac.boot_ci(x.to_numpy(), rng, mirrored=mirrored)
        out[k] = m
        out[k + " CI"] = f"[{lo:.2f}, {hi:.2f}]"
        out[k + " excl0"] = lo > 0 or hi < 0
    return out

def fig1g_seeds(beh, seeds=range(1001, 1051)):
    """Median CI bounds over 50 seeds: with seven rats a bound near zero moves with the seed."""
    out = {}
    d = {"D-U": beh.danger - beh.uncertainty, "U-S": beh.uncertainty - beh.safety, "D-S": beh.danger - beh.safety}
    for k, x in d.items():
        b = np.array([nac.boot_ci(x.to_numpy(), np.random.default_rng(s))[1:] for s in seeds])
        lo, hi = np.median(b, 0)
        out[k + " median CI"] = f"[{lo:.3f}, {hi:.3f}]"
        out[k + " excl0 (median)"] = lo > 0 or hi < 0
        out[k + " excl0 share"] = np.mean((b[:, 0] > 0) | (b[:, 1] < 0))
    return out

session_sets = {"(a) 82 with a cue-responsive unit": {u["session"] for u in R}, "(b) all": set(by_session)}
nan_readings = {"drop": "drop", "0": "zero", "1 (script)": "one"}

## fig1g-discrimination

> During the recording sessions all rats discriminated the cues, with suppression ratios highest
> to danger, intermediate to uncertainty and lowest to safety; every cue pair differed.

In [ ]:
rows = []
for sl, S in session_sets.items():
    for nl, nr in nan_readings.items():
        beh = behavior(S, nr)
        o = fig1g(beh, np.random.default_rng(1001))
        o.update(fig1g_seeds(beh))
        rows.append(dict(sessions=sl, zero_over_zero=nl, n_sessions=len(S), **o))
fig1g_table = pd.DataFrame(rows)
print(fig1g_table[["sessions", "zero_over_zero", "n_sessions", "F", "df", "p", "D-U", "U-S", "D-S"]].round(3).to_string(index=False))
print()
print(fig1g_table[["sessions", "zero_over_zero"] + [c for c in fig1g_table if "median" in c or "share" in c]].round(2).to_string(index=False))
print("expected means: D-U 0.21 +- 0.05, U-S 0.47 +- 0.12, D-S 0.69 +- 0.19; paper F(2,12) = 57.41, CIs [0.16, 0.29], [0.35, 1.03], [0.50, 1.25]")
print()
beh_82 = behavior(session_sets["(a) 82 with a cue-responsive unit"], "one")
print("per rat, 82 sessions, 0/0 = 1 (the script's handling):")
print(beh_82.round(3))
print()
print("the same per-rat differences with the bootstrap-t quantiles added without reflection (variant):")
for seed in (1, 2, 3):
    o = fig1g(beh_82, np.random.default_rng(seed), mirrored=True)
    print(f"  seed {seed}: D-U {o['D-U CI']}, U-S {o['U-S CI']}, D-S {o['D-S CI']}")
print("and with the standard studentized interval, over seeds:")
for seed in (1, 2, 3):
    o = fig1g(beh_82, np.random.default_rng(seed))
    print(f"  seed {seed}: D-U {o['D-U CI']}, U-S {o['U-S CI']}, D-S {o['D-S CI']}")

**Result: reproduced, under one reading (all sessions, 0/0 read as 0), with a CI bound near
zero.** The cue effect holds under every reading (F(2,12) from 52.6 to 57.4, p < 1e-5), and the
three mean differences are within tolerance under every reading (0.20 to 0.22, 0.47 to 0.48,
0.67 to 0.70). The danger minus uncertainty and danger minus safety CIs exclude zero under every
reading. The uncertainty minus safety CI excludes zero only with all sessions and 0/0 read as 0
(median bounds over 50 seeds [0.012, 0.599], excluding zero in 68% of seeds); under the other
readings its lower bound is below zero, because one rat (ac01) has a difference of 0.15. With the
authors' handling of 0/0 (set to 1) and the 82 sessions, F(2,12) = 57.41 and the means 0.21,
0.47 and 0.69 equal the paper's values exactly. The paper's CIs are not standard studentized
intervals of these data: they match, within resampling noise, the bootstrap-t quantiles added to
the mean without reflection ([0.36, 1.03] against the paper's [0.35, 1.03] for uncertainty minus
safety, [0.52, 1.18] against [0.50, 1.25] for danger minus safety).

## fig1h-poke-rate-unrelated and fig1h-discrimination-safety

> Baseline nose poke rate was unrelated to suppression to any cue and to overall discrimination.
> Overall discrimination ... correlated negatively with the safety suppression ratio across rats.

In [ ]:
rows = []
for sl, S in session_sets.items():
    for nl, nr in nan_readings.items():
        beh = behavior(S, nr)
        disc = beh.danger - beh.safety
        row = dict(sessions=sl, zero_over_zero=nl)
        for k, y in (("danger", beh.danger), ("uncertainty", beh.uncertainty), ("safety", beh.safety), ("D-S", disc)):
            r = stats.pearsonr(beh.baseline_pokes_per_min, y)
            row[f"poke~{k} R"] = r.statistic
            row[f"poke~{k} p"] = r.pvalue
        r = stats.pearsonr(disc, beh.safety)
        row["D-S~safety R"] = r.statistic
        row["D-S~safety p"] = r.pvalue
        rows.append(row)
fig1h_table = pd.DataFrame(rows)
print(fig1h_table.round(3).to_string(index=False))
print("paper: all |R| < 0.73 and p > 0.06 for baseline poke rate; R = -0.98, p = 1.41e-4 for D-S against safety")

**Result (fig1h-poke-rate-unrelated): reproduced, resting on the 0/0 handling.** With the
authors' handling of 0/0 ratios (set to 1, which gives the paper's F exactly) or with them left
out, no correlation of baseline poke rate has p < 0.05 in either session set (smallest p 0.26;
largest |R| 0.49). With 0/0 read as 0, baseline poke rate correlates with danger suppression in
the 82 sessions (R = 0.76, p = 0.048).

**Result (fig1h-discrimination-safety): reproduced.** R is -0.98 or -0.99 with p of 1.4e-4 or
less under every reading.

## fig2-five-clusters

> K-means clustering of the 193 cue-responsive units on six firing epochs ... gave five
> functional populations; Threat Excited (5), Threat Inhibited (26), Danger Excited (55),
> Bidirectional Valence (91) and Non-Selective (16).

Matrix readings: mean normalized firing (onset, first 1 s; late, last 5 s; per cue), under the
four normalization readings of the claims file (N1 to N4), or mean firing in Hz. Start readings:
(i) the lowest summed distance over 100 k-means++ starts; (ii) the most frequent partition over
100 starts. Clusters are named by matching their centroids to the Results' patterns. The
authors' script clusters the normalization of `cubeScriptPR25.m` with one start after `rng(1)`
(`mainAnalysis.m`, "K means clustering"); that normalization is shown as a variant (the MATLAB
random stream cannot be repeated here).

In [ ]:
norm = {}
for k, kw in nac.NORM_READINGS.items():
    norm[k] = np.array([nac.norm_methods(u, **kw)[0] for u in R])
norm["script normalization"] = np.array([nac.norm_code(u)[0] for u in R])
mats = {k: np.array([nac.epochs(t) for t in v]) for k, v in norm.items()}
mats["Hz"] = np.array([np.concatenate([[nac.window_rate(u["spikes"], u["onsets"], a, b)[nac.CUE_OF_TRIAL == c].mean()
                                        for c in range(3)] for a, b in ((0, 1), (5, 10))]) for u in R])
print({k: v.shape for k, v in mats.items()})

In [ ]:
partitions = {}
rows = []
for k, X in mats.items():
    runs = nac.kmeans_runs(X, 5, 100, seed=2000)
    best = min(runs, key=lambda r: r[1])[0]
    freq_count = collections.Counter(nac.canonical(r[0]) for r in runs)
    freq, n_freq = freq_count.most_common(1)[0]
    for start, lab in (("(i) best of 100", np.asarray(best)), ("(ii) most frequent", np.asarray(freq))):
        names, cos = nac.name_clusters(X, lab)
        pops = np.array([names[l] for l in lab])
        key = f"{k} | {start}"
        partitions[key] = pops
        cnt = collections.Counter(pops)
        rows.append(dict(matrix=k, start=start, times_found=n_freq if start.startswith("(ii)") else
                         sum(nac.canonical(r[0]) == nac.canonical(best) for r in runs),
                         **{p: cnt[p] for p in nac.POPULATIONS}, min_cosine=min(cos.values()),
                         within_tolerance=all(abs(cnt[p] - e) <= e * 0.1 for p, e in zip(nac.POPULATIONS, (5, 26, 55, 91, 16)))))
fig2_table = pd.DataFrame(rows)
print(fig2_table.round(2).to_string(index=False))
print("paper: Threat Excited 5, Threat Inhibited 26, Danger Excited 55, Bidirectional Valence 91, Non-Selective 16 (10% tolerance)")

In [ ]:
claims_rows = fig2_table[fig2_table.matrix.isin(list(nac.NORM_READINGS))]
best_row = claims_rows.loc[claims_rows.min_cosine.idxmax()]
PRIMARY = f"{best_row.matrix} | {best_row.start}"
PRIMARY_NORM = best_row.matrix
print(f"primary partition (claims-file readings, highest smallest cosine to the five patterns): {PRIMARY}, min cosine {best_row.min_cosine:.2f}")
print("centroids (danger, uncertainty, safety onset; then late cue) of each partition:")
for key, pops in partitions.items():
    X = mats[key.split(" | ")[0]]
    print(key)
    for p in nac.POPULATIONS:
        sel = pops == p
        print(f"   {p:22s} n={sel.sum():3d}  {np.round(X[sel].mean(0), 2)}")

In [ ]:
X = mats[PRIMARY_NORM]
rng_k = 2100
print(f"cluster number rule (variant): mean squared distance to centroid, best of 100 starts, {PRIMARY_NORM}")
for k in range(1, 8):
    runs = nac.kmeans_runs(X, k, 100, seed=rng_k)
    print(f"  k={k}: mean squared distance {min(r[1] for r in runs) / len(X):.3f}")
alt = np.array([nac.screen(u, "pooled")[0] for u in U])
X219 = np.array([nac.epochs(nac.norm_methods(u, **nac.NORM_READINGS[PRIMARY_NORM])[0]) for u, r in zip(U, alt) if r])
best = min(nac.kmeans_runs(X219, 5, 100, seed=2200), key=lambda r: r[1])[0]
names, _ = nac.name_clusters(X219, best)
print(f"counts on the 219 units of screen reading (a), {PRIMARY_NORM}, best of 100:", dict(collections.Counter(names[l] for l in best)))

**Result: partially reproduced.** No reading gives the paper's five counts within tolerance.
The five named patterns appear in some normalized readings: with N4 and the most frequent of
100 starts the centroids match the five patterns (smallest cosine 0.79) with counts of 6 Threat
Excited, 28 Threat Inhibited, 45 Danger Excited, 89 Bidirectional Valence and 25 Non-Selective;
N2 gives 7, 27, 44, 91 and 24. Other readings give partitions whose centroids do not show the
patterns (smallest cosine below 0.5). The solution is unstable: the best and the most frequent
partitions are each found in 1 to 3 of 100 starts. Clustering firing in Hz groups units by
firing rate and shows none of the patterns. The mean distance falls steadily from k = 1 to 7,
with no elbow at 5.

The later population claims use the partition, among the claims file's readings, whose
centroids best match the five patterns (N4, most frequent partition). This is an assumption: the
paper's own labels are not in the data set and its clustering cannot be repeated (the script
uses one MATLAB start after `rng(1)`). Each later claim also shows its result under every
partition.

In [ ]:
def pop_units(pops, names):
    return [i for i, p in enumerate(pops) if p in names]

prim = partitions[PRIMARY]
print("primary partition:", dict(collections.Counter(prim)))

## populations-by-rat

> Threat Excited units came from two of seven rats ..., Threat Inhibited from four rats, Danger
> Excited from six and Bidirectional Valence from all seven; the three main populations (172
> units) came from 79 sessions.

In [ ]:
rows = []
for key, pops in partitions.items():
    row = dict(partition=key)
    for p in ["Threat Excited", "Threat Inhibited", "Danger Excited", "Bidirectional Valence"]:
        row[p + " rats"] = len({R[i]["rat"] for i in pop_units(pops, [p])})
    main = pop_units(pops, nac.MAIN3)
    row["main3 units"] = len(main)
    row["main3 sessions"] = len({R[i]["session"] for i in main})
    row["all match"] = (row["Threat Excited rats"], row["Threat Inhibited rats"], row["Danger Excited rats"],
                        row["Bidirectional Valence rats"]) == (2, 4, 6, 7) and abs(row["main3 sessions"] - 79) <= 7.9
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))
print("paper: 2, 4, 6 and 7 rats; 172 units from 79 sessions")
te = pop_units(prim, ["Threat Excited"])
print("primary partition, Threat Excited units per rat:", dict(collections.Counter(R[i]["rat"] for i in te)))

**Result: not reproduced, under the assumed partition.** The primary partition's Threat Excited
units come from 3 rats (4 of 6 from ac04), Threat Inhibited from 5, Danger Excited from 6 and
Bidirectional Valence from 7; the three main populations (162 units) come from 77 sessions,
within tolerance of 79. No partition gives all four rat counts.

## most-units-threat

> Most cue-responsive NAcc units showed their greatest firing change to the shock-associated
> cues, danger and uncertainty.

In [ ]:
rows = []
for key, pops in partitions.items():
    four = len(pop_units(pops, ["Threat Excited", "Threat Inhibited", "Danger Excited", "Bidirectional Valence"]))
    rows.append(dict(partition=key, four_populations=four, share=four / len(pops)))
t = pd.DataFrame(rows)
print(t.round(3).to_string(index=False))
d_or_u = np.array([np.argmax(np.abs(x[[0, 1, 2]])) <= 1 for x in mats[PRIMARY_NORM]])
print(f"unit by unit ({PRIMARY_NORM}): largest absolute onset change to danger or uncertainty in {d_or_u.mean():.3f} of {len(d_or_u)} units (chance 2/3)")

**Result: reproduced.** Under every partition the four populations with differential cue
firing hold more than half of the cue-responsive units (77% to 97%; 87% under the primary
partition).

## Firing and waveform characteristics (fig3)

Baseline rate: mean firing over the 10 s before onset, all 16 trials. Coefficient of variance,
readings of the claims file: (a) inter-spike intervals within the 10-s baselines; (b) the whole
session, which in these files is the recorded part of each trial, 20 s before to 33 s after
onset (see the data check: nothing is recorded between trials), so intervals across the gaps
are left out. The authors' `cubeScriptPR25.m` pools the intervals within the 20 s before each onset
(variant). Peak-valley duration: valley to after-hyperpolarization peak of the mean waveform,
counted as `mainAnalysis.m` counts it ("Get firing characteristics"), in ms at 40 samples per
ms. The script then halves it ("HD"), which gives 0.075 to 0.3, half the 0.15 to 0.6 ms of
Figure 3C; the unhalved value spans the figure's range, and its boundary 0.3875 falls between
the modes only on that scale, so the unhalved value is used. Hartigan's dip with a p-value from 1000 uniform samples, as the authors'
`HartigansDipSignifTest(x, 1000)`.

In [ ]:
def isi_cv(u, a, b):
    isis = []
    for t in u["onsets"]:
        s = u["spikes"][(u["spikes"] > t + a) & (u["spikes"] <= t + b)]
        isis.append(np.diff(s))
    isi = np.concatenate(isis)
    return isi.std(ddof=1) / isi.mean() if len(isi) > 2 else np.nan

char = pd.DataFrame(dict(
    rate=[nac.window_rate(u["spikes"], u["onsets"], -10, 0).mean() for u in R],
    cv_a=[isi_cv(u, -10, 0) for u in R],
    cv_b=[isi_cv(u, -20, 33) for u in R],
    cv_script=[isi_cv(u, -20, 0) for u in R],
    hd=[nac.peak_valley(u["wave"]) for u in R],
    hd_script=[nac.peak_valley(u["wave"]) / 2 for u in R],
    pv_ms=[nac.peak_valley_ms(u["wave"]) for u in R]))
cube_cv = CUBE.fire[0, 0].CV[:, 0]
cv_check = [abs(cube_cv[cube_index[u["filename"] + u["neuron"]]] - c) < 1e-6 for u, c in zip(R, char.cv_script)]
print(f"cv_script equal to cube.fire.CV: {sum(cv_check)} of {len(R)}")
print(char.describe().round(3))

## fig3ab-rate-cv-unimodal

> Across the 172 units of the three main populations, baseline firing rate and coefficient of
> variance had unimodal distributions.

In [ ]:
rng_3 = np.random.default_rng(3001)
rows = []
for key, pops in partitions.items():
    main = pop_units(pops, nac.MAIN3)
    row = dict(partition=key, n=len(main))
    for col in ("rate", "cv_a", "cv_b", "cv_script"):
        x = char[col].to_numpy()[main]
        d, p = nac.dip_test(x[np.isfinite(x)], rng_3, 1000 if key == PRIMARY else 200)
        row[col + " dip"] = d
        row[col + " p"] = p
    rows.append(row)
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("paper: rate dip 0.013, p = 1.0; CV dip 0.016, p = 0.99 (n = 172). p from 1000 uniform samples for the primary partition, 200 for the others")

**Result: reproduced.** Under the primary partition (162 units) the dip test finds no
departure from unimodality for baseline rate (dip 0.015, p = 1.0) or for the coefficient of
variance under either reading (dip 0.023, p = 0.82; dip 0.024, p = 0.80), nor for the script's
20-s variant. No partition gives p < 0.05 for any of them.

## fig3c-waveform-bimodal

> Waveform peak-valley duration was bimodal across the 172 units; Threat Inhibited units were
> mostly wide (bimodal only at an uncorrected p), Danger Excited units unimodal and mostly narrow,
> and Bidirectional Valence units bimodal with about half narrow and half wide.

Boundary readings: (a) the dashed line of Figure 3C, which `halfDuration.m` sets at
(0.375 + 0.4) / 2 = 0.3875 and which sits at about 0.39 in the figure; (b) the lowest bin
between the two modes of the histogram of all main-population units (bins of 0.025).

In [ ]:
hd_all = char.hd.to_numpy()
print("peak-valley values (Figure 3C scale):", dict(sorted(collections.Counter(np.round(hd_all, 4)).items())))
print(f"the same in ms at 40 kHz (valley to peak): {np.unique(char.pv_ms).round(3)}")

def trough(x):
    edges = np.arange(0.0125, 0.8, 0.025)  # bins centred on the 0.025-ms sample grid
    h, _ = np.histogram(x, edges)
    lo_mode = np.argmax(np.where(edges[:-1] < 0.35, h, -1))
    hi_mode = np.argmax(np.where(edges[:-1] >= 0.35, h, -1))
    j = lo_mode + np.argmin(h[lo_mode:hi_mode + 1])
    return (edges[j] + edges[j + 1]) / 2

rows = []
for key, pops in partitions.items():
    main = pop_units(pops, nac.MAIN3)
    x = hd_all[main]
    nsim = 1000 if key == PRIMARY else 200
    d_all, p_all = nac.dip_test(x, rng_3, nsim)
    row = dict(partition=key, all_dip=d_all, all_p=p_all, trough=trough(x))
    for p, short in (("Threat Inhibited", "TI"), ("Danger Excited", "DE"), ("Bidirectional Valence", "BV")):
        y = hd_all[pop_units(pops, [p])]
        if len(y) < 5:
            continue
        d, pv = nac.dip_test(y, rng_3, nsim)
        row[short + " n"] = len(y)
        row[short + " dip"] = d
        row[short + " p"] = pv
        row[short + " narrow(a)"] = np.mean(y < nac.HD_BOUNDARY)
        row[short + " narrow(b)"] = np.mean(y < row["trough"])
    rows.append(row)
fig3c_table = pd.DataFrame(rows)
print(fig3c_table.round(3).to_string(index=False))
print("paper: all p = 0; TI dip 0.096 (+-0.0096), p = 0.035; DE dip 0.06 (+-0.006), p = 0.18; BV dip 0.11 (+-0.011), p = 0; BV 45-55% narrow")

**Result: reproduced under boundary reading (a), resting on the assumed partition.** Under the
primary partition the 162 units are bimodal (dip 0.114, p < 0.001). Threat Inhibited units are
bimodal (dip 0.125, p = 0.001) and mostly wide (36% narrow at 0.3875). Danger Excited units are
unimodal (dip 0.067, p = 0.12) and mostly narrow (82%). Bidirectional Valence units are bimodal
(dip 0.119, p < 0.001) with 52% narrow. The dips are within tolerance for Bidirectional Valence
(0.11 +- 0.011) and outside it for Threat Inhibited (0.096 +- 0.0096) and Danger Excited
(0.06 +- 0.006). With the trough boundary (b, 0.425) the Bidirectional Valence split is 56%
narrow, just outside 45 to 55%. The authors' script halves this measure, which would put every
unit below the 0.3875 boundary; the data match Figure 3C's scale only unhalved.

## Cue firing of the populations (fig4, fig5)

Normalized firing as in the opening comment of the claims file; the cue by interval ANOVA over
48 bins of 250 ms from 2 s before onset to offset, uncertainty collapsed; onset is the first 1 s
and late cue the last 5 s; studentized bootstrap CIs over units (1000 resamples). The
differences are those of the Methods (Population and single-unit firing analyses) and
`mainAnalysis.m` (`diff1 = a - b`, `diff2 = (a + b)/2 - c`, `diff3 = b - c`). "Danger and
uncertainty" readings: (a) the mean of the two values; (b) trials pooled, (4 danger + 8
uncertainty) / 12.

In [ ]:
def cue_stats(tr, rng):
    """tr: units x 3 cues x 200 bins. ANOVA and window differences."""
    A = nac.rm_anova2(tr[:, :, nac.ANOVA_BINS])
    on = tr[:, :, nac.ONSET].mean(2)
    late = tr[:, :, nac.LATE].mean(2)
    out = dict(n=len(tr), F_cue=A["A"][0], df_cue=f"{A['A'][1]},{A['A'][2]}", p_cue=A["A"][3],
               F_int=A["B"][0], p_int=A["B"][3], F_x=A["AxB"][0], df_x=f"{A['AxB'][1]},{A['AxB'][2]}", p_x=A["AxB"][3])
    for w, v in (("onset", on), ("late", late)):
        d, un, s = v[:, 0], v[:, 1], v[:, 2]
        diffs = {"D-U": d - un, "U-S": un - s, "(D+U)/2-S": (d + un) / 2 - s, "(4D+8U)/12-S": (4 * d + 8 * un) / 12 - s,
                 "D-(U+S)/2": d - (un + s) / 2}
        for k, x in diffs.items():
            m, lo, hi = nac.boot_ci(x, rng)
            out[f"{w} {k}"] = (round(m, 3), round(lo, 3), round(hi, 3))
    return out

def excl(ci):
    return ci[1] > 0 or ci[2] < 0

def pop_traces(pops, p, reading=PRIMARY_NORM):
    return norm[reading][pop_units(pops, [p])]

def run_pop(p, judge, seed):
    """Run cue_stats for population p under every partition (primary normalization; for the
    primary partition also every normalization reading). judge(stats) -> outcome string."""
    rows = []
    for key, pops in partitions.items():
        readings = list(nac.NORM_READINGS) + ["script normalization"] if key == PRIMARY else [PRIMARY_NORM]
        for nr in readings:
            tr = pop_traces(pops, p, nr)
            if len(tr) < 5:
                rows.append(dict(partition=key, normalization=nr, n=len(tr), outcome="too few units"))
                continue
            st = cue_stats(tr, np.random.default_rng(seed))
            rows.append(dict(partition=key, normalization=nr, **st, outcome=judge(st)))
    return pd.DataFrame(rows)

def show(t, cols):
    print(t[["partition", "normalization", "n"] + [c for c in cols if c in t] + ["outcome"]].to_string(index=False))
    print("\noutcomes:", dict(collections.Counter(t.outcome)))

## fig4ab-thi-threat-firing

> Threat Inhibited units decreased firing more to danger and uncertainty than to safety, both at
> cue onset and during late cue.

In [ ]:
def judge_4ab(st):
    ok_cis = []
    for key in ("(D+U)/2-S", "(4D+8U)/12-S"):
        on, la = st["onset " + key], st["late " + key]
        cis = excl(on) and excl(la) and on[0] < 0 and la[0] < 0
        within = abs(on[0] + 0.47) <= 0.15 and abs(la[0] + 0.44) <= 0.16
        ok_cis.append((cis, within))
    anova = st["p_cue"] < 0.05 and st["p_x"] < 0.05
    if any(c and w for c, w in ok_cis) and anova:
        return "reproduced"
    if any(c for c, w in ok_cis):
        return "partially_reproduced"
    return "not_reproduced"

t4ab = run_pop("Threat Inhibited", judge_4ab, 4001)
show(t4ab, ["F_cue", "df_cue", "p_cue", "F_x", "df_x", "p_x", "onset (D+U)/2-S", "late (D+U)/2-S", "onset (4D+8U)/12-S", "late (4D+8U)/12-S"])
print("paper: F(2,50) = 37.84, interaction F(94,2350) = 4.95; onset -0.47 [-0.63, -0.32], late -0.44 [-0.60, -0.20]")

**Result: partially reproduced, resting on the assumed partition.** Under the primary
partition (28 Threat Inhibited units) the cue effect (F(2,54) = 41.8) and the interaction
(F(94,2538) = 3.83) have p < 1e-10, and threat minus safety is below zero at onset (-0.72
[-1.04, -0.42] with N4) and in the late cue (-0.70 [-0.98, -0.48]) under every normalization
reading, but the means are larger than the paper's (-0.47 +- 0.15, -0.44 +- 0.16) under every
claims-file reading. With the authors' script normalization (variant) they are -0.37 and -0.38,
within tolerance.

## fig4b-thi-danger-equals-uncertainty

> Threat Inhibited firing decreases to danger and uncertainty did not differ, at onset or during
> late cue.

In [ ]:
def judge_4b(st):
    return "reproduced" if not excl(st["onset D-U"]) and not excl(st["late D-U"]) else "not_reproduced"

t4b = run_pop("Threat Inhibited", judge_4b, 4101)
show(t4b, ["onset D-U", "late D-U"])
print("paper: onset -0.01 [-0.11, 0.12], late -0.02 [-0.14, 0.12]")

**Result: reproduced, resting on the assumed partition.** Danger minus uncertainty CIs contain
zero at onset (-0.03 [-0.26, 0.22]) and in the late cue (-0.13 [-0.35, 0.07]) under the
primary partition, under every normalization reading. Under three partitions found with N1 to
N3 and the best of 100 starts, danger decreases exceed uncertainty decreases.

## fig4cd-de-danger-firing

> Danger Excited units fired most to danger and discriminated all three cues at onset, and
> during late cue their firing was selective for danger over uncertainty and safety.

In [ ]:
def judge_4cd(st):
    cis = [st["onset D-U"], st["onset U-S"], st["late D-(U+S)/2"]]
    exp = [(0.22, 0.11), (0.21, 0.12), (0.17, 0.10)]
    allx = all(excl(c) and c[0] > 0 for c in cis)
    within = all(abs(c[0] - e) <= t for c, (e, t) in zip(cis, exp))
    if allx and within and st["p_cue"] < 0.05 and st["p_x"] < 0.05:
        return "reproduced"
    return "partially_reproduced" if allx else "not_reproduced"

t4cd = run_pop("Danger Excited", judge_4cd, 4201)
show(t4cd, ["F_cue", "df_cue", "p_cue", "F_x", "p_x", "onset D-U", "onset U-S", "late D-(U+S)/2"])
print("paper: F(2,106) = 13.05, interaction F(94,4982) = 2.23; onset D-U 0.22 [0.11, 0.33], U-S 0.21 [0.07, 0.33]; late D-(U+S)/2 0.17 [0.07, 0.27]")

**Result: partially reproduced, resting on the assumed partition.** Under the primary
partition (45 Danger Excited units) the cue effect (F(2,88) = 12.7, p = 1.4e-5) and the
interaction (p = 2.3e-8) hold, and all three CIs exclude zero (onset danger minus uncertainty
0.40 [0.12, 0.64], uncertainty minus safety 0.66 [0.41, 0.93], late danger minus the others 0.30
[0.12, 0.50] with N4), but every mean is above its tolerance under the claims-file readings. With
the script normalization (variant) all three are within tolerance (0.24, 0.25, 0.16).

## fig4ef-bv-danger-decrease

> Bidirectional Valence units, the most abundant population (about 47% of cue-responsive
> units), decreased firing to danger more than to uncertainty and safety, at onset and during
> late cue.

In [ ]:
def judge_4ef(st, share):
    on, la = st["onset D-(U+S)/2"], st["late D-(U+S)/2"]
    allx = excl(on) and excl(la) and on[0] < 0 and la[0] < 0
    within = abs(on[0] + 0.13) <= 0.09 and abs(la[0] + 0.09) <= 0.07 and abs(share - 47) <= 4.7
    if allx and within and st["p_cue"] < 0.05 and st["p_x"] < 0.05:
        return "reproduced"
    return "partially_reproduced" if allx else "not_reproduced"

rows = []
for key, pops in partitions.items():
    share = 100 * np.mean(pops == "Bidirectional Valence")
    readings = list(nac.NORM_READINGS) + ["script normalization"] if key == PRIMARY else [PRIMARY_NORM]
    for nr in readings:
        st = cue_stats(pop_traces(pops, "Bidirectional Valence", nr), np.random.default_rng(4301))
        rows.append(dict(partition=key, normalization=nr, share=share, **st, outcome=judge_4ef(st, share)))
t4ef = pd.DataFrame(rows)
show(t4ef.round(3), ["share", "F_cue", "df_cue", "p_cue", "F_x", "p_x", "onset D-(U+S)/2", "late D-(U+S)/2"])
print("paper: 47% of cue-responsive units; F(2,178) = 8.90, interaction F(94,8366) = 1.45; onset -0.13 [-0.22, -0.03], late -0.09 [-0.16, -0.02]")

**Result: partially reproduced, resting on the assumed partition.** Bidirectional Valence units
are 46.1% of cue-responsive units (within 47 +- 4.7); the cue effect (F(2,176) = 18.4) and the
interaction (p = 0.001) hold; danger minus uncertainty and safety is below zero at onset (-0.33
[-0.48, -0.16]) and in the late cue (-0.20 [-0.32, -0.06]) under every normalization reading,
with means beyond the tolerance (-0.13 +- 0.09, -0.09 +- 0.07). With the script normalization
the onset mean is -0.15 and the late CI reaches zero ([-0.15, 0.00]).

## fig4f-bv-safety-variability

> Bidirectional Valence units varied more in their safety onset firing than in their danger or
> uncertainty onset firing.

Pitman-Morgan test of paired variances, as the authors' `PitmanMorganTest` in `mainAnalysis.m`.

In [ ]:
rows = []
for key, pops in partitions.items():
    readings = list(nac.NORM_READINGS) + ["script normalization"] if key == PRIMARY else [PRIMARY_NORM]
    for nr in readings:
        on = pop_traces(pops, "Bidirectional Valence", nr)[:, :, nac.ONSET].mean(2)
        r1 = nac.pitman_morgan(on[:, 2], on[:, 0])
        r2 = nac.pitman_morgan(on[:, 2], on[:, 1])
        ok = r1[2] < 0.05 and r2[2] < 0.05 and r1[0] > 1 and r2[0] > 1
        rows.append(dict(partition=key, normalization=nr, n=len(on), var_ratio_S_D=r1[0], p_S_D=r1[2],
                         var_ratio_S_U=r2[0], p_S_U=r2[2], outcome="reproduced" if ok else "not_reproduced"))
t4f = pd.DataFrame(rows)
show(t4f.round(4), ["var_ratio_S_D", "p_S_D", "var_ratio_S_U", "p_S_U"])
print("paper: p = 0.0087 (safety against danger), 0.0046 (safety against uncertainty), n = 91")

**Result: reproduced, resting on the assumed partition.** Under the primary partition and every
normalization reading of the claims file, safety onset firing varies more than danger onset
firing (variance ratio 1.68 to 1.73, p 0.010 to 0.015) and more than uncertainty onset firing
(ratio 1.69 to 1.84, p 0.005 to 0.015). With the script normalization (variant) the safety
against danger test is not significant (ratio 1.25, p = 0.29).

## fig5ab-the-onset

> Threat Excited units increased firing more to danger and uncertainty than to safety at onset,
> but not during late cue.

In [ ]:
def judge_5ab(st):
    res = []
    for key in ("(D+U)/2-S", "(4D+8U)/12-S"):
        on, la = st["onset " + key], st["late " + key]
        res.append(st["p_x"] < 0.05 and excl(on) and on[0] > 0 and abs(on[0] - 0.9) <= 0.6 and not excl(la))
    return "reproduced" if any(res) else "not_reproduced"

t5ab = run_pop("Threat Excited", judge_5ab, 5001)
show(t5ab, ["F_int", "F_x", "df_x", "p_x", "onset (D+U)/2-S", "late (D+U)/2-S", "onset (4D+8U)/12-S", "late (4D+8U)/12-S"])
print("paper: n = 5; interval F(47,188) = 19.75, interaction F(94,376) = 1.46, p = 0.008; onset 0.9 [0.30, 12.58]; late [-1.08, 1.11]")

**Result: not reproduced, under the assumed partition.** The primary partition has 6 Threat
Excited units. Threat minus safety firing is above zero at onset (1.38 [0.33, 2.36] with N4) and
its late CI contains zero, but the cue by interval interaction is not significant under any
normalization reading (p 0.08 to 0.11).

## fig5cd-ns-nonselective

> Non-Selective units showed no main effect of cue and no cue by interval interaction, and their
> threat and safety firing did not differ at onset or late cue.

In [ ]:
def judge_5cd(st):
    # a claim of absence, judged on the paper's tests; reading (a) of "danger and uncertainty"
    ok = st["p_cue"] >= 0.05 and st["p_x"] >= 0.05 and not excl(st["onset (D+U)/2-S"]) and not excl(st["late (D+U)/2-S"])
    return "reproduced" if ok else "not_reproduced"

t5cd = run_pop("Non-Selective", judge_5cd, 5101)
show(t5cd, ["F_cue", "df_cue", "p_cue", "F_x", "p_x", "onset (D+U)/2-S", "late (D+U)/2-S"])
print("paper: n = 16; F(2,30) = 3.04, p = 0.063; interaction F(94,1410) = 1.19, p = 0.11; onset [-0.22, 0.36], late [-0.09, 0.46]")

**Result: not reproduced, under the assumed partition.** The 25 Non-Selective units of the
primary partition show a main effect of cue (F(2,48) = 4.9 to 5.2, p 0.009 to 0.012), and their
late threat minus safety CI excludes zero (0.27 to 0.29, lower bounds 0.01 to 0.03); the onset
CI contains zero. The interaction is at p 0.04 to 0.07.

## Firing similarity (fig6)

Per unit, mean normalized firing to each cue in 12 bins of 1 s from 2 s before onset to offset;
Pearson R between the traces of each cue pair. Readings: (a) the mean of four 250-ms normalized
bins; (b) 1-s bins normalized in the same way. The authors' `mainAnalysis.m` correlates the 1-s
bins of the script normalization (columns 19 to 30 of `fire.z.s1`), shown as a variant. Units
with a constant trace (no R) are left out and counted.

In [ ]:
def similarity(pops, reading, nr=PRIMARY_NORM):
    idx = pop_units(pops, nac.MAIN3)
    out = []
    for i in idx:
        u = R[i]
        if reading == "a":
            tr = norm[nr][i][:, nac.ONSET_BIN - 8: nac.ONSET_BIN + 40].reshape(3, 12, 4).mean(2)
        elif reading == "b":
            kw = nac.NORM_READINGS[nr] if nr in nac.NORM_READINGS else {}
            tr = nac.norm_methods(u, bin_s=1.0, **kw)[0][:, 18:30]
        else:
            z = nac.trial_code(u, 1.0)
            tr = np.array([z[nac.CUE_OF_TRIAL == c].mean(0) for c in range(3)])[:, 18:30]
        rr = [stats.pearsonr(tr[0], tr[1]).statistic if tr[0].std() > 0 and tr[1].std() > 0 else np.nan,
              stats.pearsonr(tr[1], tr[2]).statistic if tr[1].std() > 0 and tr[2].std() > 0 else np.nan,
              stats.pearsonr(tr[0], tr[2]).statistic if tr[0].std() > 0 and tr[2].std() > 0 else np.nan]
        out.append(dict(pop=pops[i], DU=rr[0], US=rr[1], DS=rr[2]))
    return pd.DataFrame(out)

sim = {}
for key, pops in partitions.items():
    for rd in ("a", "b", "script"):
        if key != PRIMARY and rd == "script":
            continue
        sim[(key, rd)] = similarity(pops, rd)

## fig6-similarity-anova

> Temporal firing similarity between cue pairs depended on population and comparison, with a
> population by comparison interaction.

In [ ]:
rows = []
for (key, rd), s in sim.items():
    s = s.dropna()
    groups = [s[s["pop"] == p][["DU", "US", "DS"]].to_numpy() for p in nac.MAIN3]
    A = nac.mixed_anova(groups)
    allp = all(A[k][3] < 0.05 for k in A)
    out = "reproduced" if allp else ("partially_reproduced" if A["group x level"][3] < 0.05 else "not_reproduced")
    rows.append(dict(partition=key, reading=rd, n=len(s), dropped=len(sim[(key, rd)]) - len(s),
                     F_pop=A["group"][0], df_pop=f"{A['group'][1]},{A['group'][2]}", p_pop=A["group"][3],
                     F_comp=A["level"][0], p_comp=A["level"][3], F_x=A["group x level"][0],
                     df_x=f"{A['group x level'][1]},{A['group x level'][2]}", p_x=A["group x level"][3], outcome=out))
t6 = pd.DataFrame(rows)
print(t6.round(4).to_string(index=False))
print("paper: population F(2,169) = 3.34, p = 0.038; comparison F(2,338) = 26.26; interaction F(4,338) = 3.86, p = 0.004")
print("outcomes:", dict(collections.Counter(t6.outcome)))

**Result: reproduced, resting on the assumed partition.** Under the primary partition (162
units, none dropped) population (F(2,159) = 5.66, p = 0.004), comparison (F(2,318) = 12.0, p <
1e-4) and their interaction (F(4,318) = 2.45, p = 0.046) are significant. Readings (a) and (b)
give identical R values: the 1-s normalization differs from the averaged 250-ms one by a scale
and an offset per unit, which leave Pearson's R unchanged.

## fig6d-thi-similarity, fig6e-de-similarity, fig6f-bv-no-similarity-difference

Paired t tests between comparisons within each population, at the Bonferroni level 0.05 / 9.

In [ ]:
rows = []
for (key, rd), s in sim.items():
    s = s.dropna()
    row = dict(partition=key, reading=rd)
    for p, short in (("Threat Inhibited", "TI"), ("Danger Excited", "DE"), ("Bidirectional Valence", "BV")):
        x = s[s["pop"] == p]
        for a, b in (("DU", "US"), ("DU", "DS"), ("US", "DS")):
            t = stats.ttest_rel(x[a], x[b])
            row[f"{short} {a}-{b}"] = (round(t.statistic, 2), float(f"{t.pvalue:.2g}"))
        row[short + " n"] = len(x)
    row["fig6d"] = "reproduced" if all(row[f"TI DU-{c}"][0] > 0 and row[f"TI DU-{c}"][1] < 0.05 / 9 for c in ("US", "DS")) else "not_reproduced"
    row["fig6e"] = "reproduced" if all(row[f"DE DU-{c}"][0] > 0 and row[f"DE DU-{c}"][1] < 0.05 / 9 for c in ("US", "DS")) else "not_reproduced"
    row["fig6f"] = "reproduced" if all(row[f"BV {c}"][1] >= 0.05 / 9 for c in ("DU-US", "DU-DS", "US-DS")) else "not_reproduced"
    rows.append(row)
t6t = pd.DataFrame(rows)
print(t6t.to_string(index=False))
print("paper: TI t(25) = 3.31 and 4.11; DE t(54) = 3.24 and 4.21; BV all t < 2.1, p > 0.0056")
for c in ("fig6d", "fig6e", "fig6f"):
    print(c, dict(collections.Counter(t6t[c])))

**Result (fig6d-thi-similarity): reproduced, resting on the assumed partition.** Threat
Inhibited danger/uncertainty R exceeds uncertainty/safety (t(27) = 3.08, p = 0.0047) and
danger/safety (t(27) = 3.49, p = 0.0017), both below 0.0056.

**Result (fig6e-de-similarity): not reproduced, under the assumed partition.** Danger Excited
danger/uncertainty R is higher than both other pairs (t(44) = 2.36, p = 0.023; t(44) = 2.64,
p = 0.011) but neither passes 0.0056. With the script normalization (variant) both pass
(p = 0.004 and 0.0014).

**Result (fig6f-bv-no-similarity-difference): reproduced, resting on the assumed partition.**
No Bidirectional Valence comparison reaches 0.0056 (largest t 1.81, p = 0.074).

## fig6-thi-greatest-similarity

> Danger/uncertainty firing similarity in Threat Inhibited units exceeded that of Danger Excited
> and Bidirectional Valence units, and was the greatest of all populations and cue pairs.

In [ ]:
rows = []
for (key, rd), s in sim.items():
    s = s.dropna()
    means = s.groupby("pop")[["DU", "US", "DS"]].mean()
    ti = s[s["pop"] == "Threat Inhibited"].DU
    t1 = stats.ttest_ind(ti, s[s["pop"] == "Danger Excited"].DU)
    t2 = stats.ttest_ind(ti, s[s["pop"] == "Bidirectional Valence"].DU)
    greatest = means.stack().idxmax() == ("Threat Inhibited", "DU")
    ok = [t1.statistic > 0 and t2.statistic > 0 and t1.pvalue < lvl and t2.pvalue < lvl and greatest for lvl in (0.05, 0.05 / 9)]
    rows.append(dict(partition=key, reading=rd, TI_DU=means.loc["Threat Inhibited", "DU"], max_cell=str(means.stack().idxmax()),
                     t_vs_DE=t1.statistic, p_vs_DE=t1.pvalue, t_vs_BV=t2.statistic, p_vs_BV=t2.pvalue,
                     outcome="reproduced" if any(ok) else "not_reproduced"))
t6g = pd.DataFrame(rows)
print(t6g.round(4).to_string(index=False))
print("outcomes:", dict(collections.Counter(t6g.outcome)))

**Result: reproduced, resting on the assumed partition.** Threat Inhibited danger/uncertainty R
(mean 0.52) is the highest of the nine population and pair means, and exceeds Danger Excited
(t = 3.59, p = 0.0006) and Bidirectional Valence (t = 4.20, p = 0.0001), at either level.

## fig7-tuning-curves

> The extreme of the Threat Inhibited tuning curve was at an uncertainty assignment of 0.75, and
> those of Danger Excited and Bidirectional Valence units at 0.50 ...

Per unit, the 16 trials of its session; the cue regressor 1 for danger, 0 for safety and u for
the eight uncertainty trials; linear regression with a constant (Methods, Tuning curve). Per-trial
normalization readings (a), (b), (c) of the claims file, in 1-s bins. The regression readings (a)
one regression on each trial's 10-s cue mean and (b) the mean of the ten 1-s betas give the same
beta, since the beta is linear in the response; both are computed and compared. The authors'
`cubeScriptPR25.m` and `mainAnalysis.m` regress the script's per-trial normalization on the cue
regressor and the trial's suppression ratio together, and `probability_1.m` averages the betas
over the cue's 1-s bins (columns 3 to 12); that is shown as a variant.

In [ ]:
U_LEVELS = (0.0, 0.25, 0.5, 0.75, 1.0)
CUE_1S = slice(20, 30)

def tuning(pops, p, reading, nr=PRIMARY_NORM, with_sr=False):
    betas_a, betas_b = [], []
    kw = nac.NORM_READINGS.get(nr, {})
    for i in pop_units(pops, [p]):
        u = R[i]
        if reading == "script":
            y = nac.trial_code(u, 1.0)
        else:
            _, sd, _ = nac.norm_methods(u, **kw)
            y, _ = nac.trial_norm(u, reading, sd)
        sr, _ = nac.suppression(u, "one")
        ba, bb = [], []
        for lv in U_LEVELS:
            X = np.column_stack([nac.pattern(lv), sr]) if with_sr else nac.pattern(lv)[:, None]
            ba.append(nac.ols_betas(y[:, CUE_1S].mean(1), X)[0])
            bb.append(np.mean(nac.ols_betas(y[:, CUE_1S], X)[0]))
        betas_a.append(ba)
        betas_b.append(bb)
    return np.array(betas_a), np.array(betas_b)

In [ ]:
expected_ext = {"Threat Inhibited": 0.75, "Danger Excited": 0.5, "Bidirectional Valence": 0.5}
rows = []
extremes = {}
for key, pops in partitions.items():
    for rd in ("a", "b", "c", "script"):
        if rd == "script" and key != PRIMARY:
            continue
        row = dict(partition=key, normalization=rd)
        hits = 0
        for p in nac.MAIN3:
            ba, bb = tuning(pops, p, rd, with_sr=(rd == "script"))
            mb = np.nanmean(ba, 0)
            ext = U_LEVELS[int(np.nanargmax(np.abs(mb)))]
            row[p + " betas"] = np.round(mb, 3).tolist()
            row[p + " extreme"] = ext
            row[p + " max |a-b|"] = float(np.nanmax(np.abs(ba - bb)))
            hits += ext == expected_ext[p]
            extremes[(key, rd, p)] = ext
        row["matches"] = hits
        row["outcome"] = "reproduced" if hits == 3 else ("partially_reproduced" if hits == 2 else "not_reproduced")
        rows.append(row)
t7 = pd.DataFrame(rows)
print(t7.drop(columns=[c for c in t7 if c.endswith("|a-b|")]).to_string(index=False))
print(f"largest difference between regression readings (a) and (b): {t7[[c for c in t7 if c.endswith('|a-b|')]].to_numpy().max():.2e}")
print("paper: Threat Inhibited 0.75, Danger Excited 0.50, Bidirectional Valence 0.50")
print("outcomes:", dict(collections.Counter(t7.outcome)))

**Result: reproduced, resting on the assumed partition.** Under per-trial normalization
readings (a) and (b) the extremes are 0.75 for Threat Inhibited, and 0.50 for Danger Excited
and Bidirectional Valence, as the paper reports; the two regression readings give the same
betas (largest difference below 1e-14). The Threat Inhibited extreme is narrow: its mean betas at
0.50 and 0.75 are -1.013 and -1.020 under reading (a). Under reading (c) the Threat Inhibited
extreme is 0.50. The script's procedure (variant) gives 0.75, 0.50 and 0.50.

## Fear output and exaggerated threat (fig8)

Per unit and 1-s bin (2 s before onset to offset, 12 bins), per-trial normalized firing over the
16 trials regressed on the trial's suppression ratio (fear output) and the cue pattern with the
population's uncertainty value (0.75, 0.50, 0.50, as the criterion gives), with a constant.
Readings: (a) one model with both regressors; (b) a separate model for each. Per-trial
normalization readings (a), (b), (c); 0/0 suppression ratios left out (trial dropped) or set to
0, and set to 1 as the scripts do (variant). Early cue is the first 5 s, late cue the last 5 s.
Units whose suppression ratio does not vary are left out. Repeated-measures ANOVA on beta,
regressor by bin.

In [ ]:
BINS12 = slice(18, 30)

def fig8(pops, p, uval, norm_reading, model, nan_reading, nr=PRIMARY_NORM):
    kw = nac.NORM_READINGS.get(nr, {})
    B = []
    dropped = 0
    for i in pop_units(pops, [p]):
        u = R[i]
        if norm_reading == "script":
            y = nac.trial_code(u, 1.0)
        else:
            _, sd, _ = nac.norm_methods(u, **kw)
            y, _ = nac.trial_norm(u, norm_reading, sd)
        y = y[:, BINS12]
        sr, _ = nac.suppression(u, nan_reading)
        keep = np.isfinite(sr)
        if keep.sum() < 4 or np.nanstd(sr[keep]) == 0 or not np.all(np.isfinite(y)):
            dropped += 1
            continue
        pat = nac.pattern(uval)
        if model == "joint":
            b = nac.ols_betas(y[keep], np.column_stack([sr[keep], pat[keep]]))
            fo, et = b[0], b[1]
        else:
            fo = nac.ols_betas(y[keep], sr[keep][:, None])[0]
            et = nac.ols_betas(y[keep], pat[keep][:, None])[0]
        if not (np.all(np.isfinite(fo)) and np.all(np.isfinite(et))):
            dropped += 1
            continue
        B.append(np.stack([fo, et]))
    return np.array(B), dropped

def fig8_stats(B, rng):
    A = nac.rm_anova2(B)
    early, late = slice(2, 7), slice(7, 12)
    out = dict(n=len(B), F_reg=A["A"][0], df_reg=f"{A['A'][1]},{A['A'][2]}", p_reg=A["A"][3],
               F_x=A["AxB"][0], df_x=f"{A['AxB'][1]},{A['AxB'][2]}", p_x=A["AxB"][3])
    for w, sl in (("early", early), ("late", late)):
        fo, et = B[:, 0, sl].mean(1), B[:, 1, sl].mean(1)
        for k, x in (("ET", et), ("FO", fo), ("ET-FO", et - fo)):
            m, lo, hi = nac.boot_ci(x, rng)
            out[f"{w} {k}"] = (round(m, 3), round(lo, 3), round(hi, 3))
    return out

def run_fig8(p, uval, judge, seed):
    rows = []
    for key, pops in partitions.items():
        for nrd in ("a", "b", "c", "script"):
            for model in ("joint", "separate"):
                for nan_r in ("drop", "zero", "one"):
                    if key != PRIMARY and not (nrd == "a" and model == "joint" and nan_r == "drop"):
                        continue
                    if nrd == "script" and not (model == "joint" and nan_r == "one"):
                        continue
                    B, dropped = fig8(pops, p, uval, nrd, model, nan_r)
                    if len(B) < 3:
                        rows.append(dict(partition=key, norm=nrd, model=model, zero_over_zero=nan_r, n=len(B), outcome="too few units"))
                        continue
                    st = fig8_stats(B, np.random.default_rng(seed))
                    rows.append(dict(partition=key, norm=nrd, model=model, zero_over_zero=nan_r, dropped=dropped, **st, outcome=judge(st)))
    t = pd.DataFrame(rows)
    return t

def show8(t, cols):
    print(t[["partition", "norm", "model", "zero_over_zero", "n", "dropped"] + cols + ["outcome"]].to_string(index=False))
    claims_readings = t[t.norm != "script"]
    print("\noutcomes, claims-file readings:", dict(collections.Counter(claims_readings.outcome)),
          "| primary partition:", dict(collections.Counter(claims_readings[claims_readings.partition == PRIMARY].outcome)))

## fig8ab-thi-exaggerated-threat

> Threat Inhibited firing signaled exaggerated threat and not fear output; exaggerated threat
> betas were below zero at early and late cue, and differed from fear output betas at early but
> not late cue.

In [ ]:
def judge_8ab(st):
    e, l, fe, fl, de, dl = (st[k] for k in ("early ET", "late ET", "early FO", "late FO", "early ET-FO", "late ET-FO"))
    core = excl(e) and excl(l) and e[0] < 0 and l[0] < 0 and not excl(fe) and not excl(fl)
    full = (core and st["p_x"] < 0.05 and excl(de) and not excl(dl) and abs(e[0] + 0.97) <= 0.41
            and abs(l[0] + 0.47) <= 0.39 and abs(de[0] + 0.92) <= 0.6)
    return "reproduced" if full else ("partially_reproduced" if core else "not_reproduced")

t8ab = run_fig8("Threat Inhibited", 0.75, judge_8ab, 8001)
show8(t8ab, ["F_x", "df_x", "p_x", "early ET", "late ET", "early FO", "late FO", "early ET-FO", "late ET-FO"])
print("paper: interaction F(11,264) = 2.32, p = 0.01; ET early -0.97 [-1.42, -0.56], late -0.47 [-0.86, -0.05]; ET-FO early -0.92 [-1.58, -0.32], late [-1.06, 0.43]")

**Result: reproduced, under one reading and the assumed partition.** With per-trial reading (a),
one model and 0/0 trials left out (27 units, 1 dropped for a constant suppression ratio), the
interaction is F(11,286) = 1.88, p = 0.041; exaggerated threat betas are -1.15 [-1.63, -0.77]
early and -0.57 [-1.04, -0.17] late; fear output CIs contain zero; the early difference is -0.98
[-1.70, -0.29] and the late difference CI contains zero, all within tolerance. The same holds with
0/0 set to 1. With separate models, fear output betas are themselves below zero, and the claim
fails; under readings (b) and (c) the early fear output CI excludes zero.

## fig8cd-de-exaggerated-threat

> Danger Excited firing signaled exaggerated threat and not fear output; exaggerated threat betas
> exceeded zero at early and late cue, and exceeded fear output betas at early but not late cue.

In [ ]:
def judge_8cd(st):
    e, l, fe, fl, de, dl = (st[k] for k in ("early ET", "late ET", "early FO", "late FO", "early ET-FO", "late ET-FO"))
    core = excl(e) and excl(l) and e[0] > 0 and l[0] > 0 and not excl(fe) and not excl(fl)
    full = (core and st["p_reg"] < 0.05 and excl(de) and de[0] > 0 and not excl(dl) and abs(e[0] - 0.43) <= 0.18
            and abs(l[0] - 0.3) <= 0.21 and abs(de[0] - 0.39) <= 0.29)
    return "reproduced" if full else ("partially_reproduced" if core else "not_reproduced")

t8cd = run_fig8("Danger Excited", 0.5, judge_8cd, 8101)
show8(t8cd, ["F_reg", "df_reg", "p_reg", "early ET", "late ET", "early FO", "late FO", "early ET-FO", "late ET-FO"])
print("paper: regressor F(1,53) = 6.64, p = 0.01; ET early 0.43 [0.22, 0.61], late 0.30 [0.06, 0.51]; ET-FO early 0.39 [0.10, 0.69], late [-0.05, 0.58]")

**Result: partially reproduced, resting on the assumed partition.** With one model (45 units),
exaggerated threat betas are above zero early (0.47 [0.16, 0.73]) and late (0.28 [0.03, 0.55])
and fear output CIs contain zero, but the early difference CI contains zero (0.33 [-0.12,
0.64]) under every one-model reading, and the main effect of regressor (p = 0.13 here) is
significant in only two of them. With separate models the fear output betas exceed zero.

## fig8ef-bv-exaggerated-threat

> Bidirectional Valence firing signaled exaggerated threat and not fear output; early
> exaggerated threat betas were below zero while fear output betas were not, and the two
> differed at early but not late cue.

In [ ]:
def judge_8ef(st):
    e, fe, de, dl = (st[k] for k in ("early ET", "early FO", "early ET-FO", "late ET-FO"))
    full = (st["p_reg"] < 0.05 and st["p_x"] < 0.05 and excl(e) and e[0] < 0 and excl(de) and de[0] < 0
            and abs(e[0] + 0.31) <= 0.14 and abs(de[0] + 0.39) <= 0.23 and not excl(fe) and not excl(dl))
    core = excl(e) and e[0] < 0 and not excl(fe)
    return "reproduced" if full else ("partially_reproduced" if core else "not_reproduced")

t8ef = run_fig8("Bidirectional Valence", 0.5, judge_8ef, 8201)
show8(t8ef, ["F_reg", "p_reg", "F_x", "df_x", "p_x", "early ET", "early FO", "early ET-FO", "late ET-FO"])
print("paper: regressor F(1,85) = 6.22, p = 0.015; interaction F(11,935) = 2.56, p = 0.003; ET early -0.31 [-0.45, -0.16]; FO early [-0.05, 0.20]; ET-FO early -0.39 [-0.62, -0.12], late [-0.40, 0.08]")

**Result: partially reproduced, resting on the assumed partition.** With reading (a), one model
and 0/0 trials left out (80 units, 9 dropped), the regressor effect (p = 0.025) and the
interaction (p = 0.049) hold, early exaggerated threat betas are below zero (-0.47 [-0.64,
-0.29]) while fear output betas are not, and the early difference (-0.46 [-0.74, -0.17]) holds
with the late one containing zero; only the early exaggerated threat mean is outside its
tolerance (-0.31 +- 0.14). The script's procedure (variant) meets every part (-0.30 [-0.47,
-0.14]).

## Reward and nose poke cessation firing (fig9, fig10)

Pellet feeder advances (EVT02) and nose pokes followed by at least 2 s without a poke, in the
ITI. ITI readings of the claims file: (a) the event's 4-s window outside 20 s before onset to 20 s
after offset; (b) the window overlapping no cue and no shock. Normalization readings: (i) the
event-averaged trace minus its 1-s pre-event mean, divided by the unit's SD from the cue
normalization (N1); (ii) the same trace z-scored over its 16 bins. The authors' script
(`cubeScriptPR25.m`) takes events 18 to 2 s before onset or 14 to 28 s after it (pellets), and
18 to 2 s before onset (cessation), and z-scores per event; shown as a variant. Units with no
event are left out. Pre is the 250-ms bin before the event, post the first bin after it.

The files hold nothing more than 33.5 s after an onset (data check), so ITI reading (a) leaves
at most one reward event per unit, and is shown but cannot support the analysis. The ITI window
and the event normalization are details the Methods leave open and the authors' script settles,
so the script's reading is run as a reading taken from the paper's code (author, step 1.6).
The cue firing used in the correlations is the claims file's normalization.

In [ ]:
sd_n1 = np.array([nac.norm_methods(u, **nac.NORM_READINGS[PRIMARY_NORM])[1] for u in R])
EVENT_READINGS = [("a", "sd"), ("a", "z16"), ("b", "sd"), ("b", "z16"), ("script", "code")]

def event_traces(kind):
    out = {}
    for itr, nm in EVENT_READINGS:
        tr, n_ev = [], []
        for i, u in enumerate(R):
            if kind == "pellet":
                ev = nac.iti_events_code(u["pellet"], u) if itr == "script" else nac.iti_events(u["pellet"], u, itr)
            else:
                ev = nac.cessations(u, "code" if itr == "script" else itr)
            n_ev.append(len(ev))
            tr.append(nac.event_trace(u, ev, nm, sd_n1[i]))
        out[(itr, nm)] = (np.array(tr), np.array(n_ev))
    return out

reward = event_traces("pellet")
cess = event_traces("cessation")
for name, d in (("reward", reward), ("cessation", cess)):
    for k, (tr, n_ev) in d.items():
        print(f"{name} {k}: units with events {np.sum(np.isfinite(tr[:, 0]))} of {len(tr)}, events per unit median {np.median(n_ev):.0f} (range {n_ev.min()} to {n_ev.max()})")

def event_anova(tr, pops):
    groups = []
    for p in nac.MAIN3:
        x = tr[pop_units(pops, [p])]
        groups.append(x[np.all(np.isfinite(x), 1)])
    A = nac.mixed_anova(groups)
    per = {p: nac.rm_anova1(g) for p, g in zip(nac.MAIN3, groups)}
    return A, per, groups

## fig9a-reward-firing

> Around pellet feeder advance in the ITI, Bidirectional Valence units selectively increased
> firing after reward, while Threat Inhibited and Danger Excited units did not change firing.

In [ ]:
rows = []
for key, pops in partitions.items():
    for k, (tr, _) in reward.items():
        if key != PRIMARY and k != ("a", "sd"):
            continue
        A, per, groups = event_anova(tr, pops)
        ok = (A["group x level"][3] < 0.05 and per["Bidirectional Valence"][3] < 0.05
              and per["Threat Inhibited"][3] >= 0.05 and per["Danger Excited"][3] >= 0.05)
        rows.append(dict(partition=key, reading=k, n=sum(len(g) for g in groups),
                         F_bin=A["level"][0], df_bin=f"{A['level'][1]},{A['level'][2]}", p_bin=A["level"][3],
                         F_x=A["group x level"][0], p_x=A["group x level"][3],
                         **{f"{p[:2]} F,p": (round(per[p][0], 2), float(f"{per[p][3]:.2g}")) for p in nac.MAIN3},
                         outcome="reproduced" if ok else "not_reproduced"))
t9a = pd.DataFrame(rows)
print(t9a.round(4).to_string(index=False))
print("paper: interval F(15,2430) = 3.83; interaction F(30,2430) = 1.72, p = 0.009; BV F(15,1290) = 8.78; TI and DE F < 1.1, p > 0.4")
print("outcomes:", dict(collections.Counter(t9a.outcome)))

**Result: not reproduced, under the assumed partition.** Under every reading Bidirectional
Valence units change firing around reward (script reading F(15,1320) = 8.4, p < 1e-4) and the
population by bin interaction is significant (p = 0.0007), but Threat Inhibited units also
change (F = 2.51, p = 0.0014) under the script reading, and under ITI reading (b) one of Threat
Inhibited or Danger Excited does too.

## fig9bcd-reward-pre-post

> Bidirectional Valence firing around zero before reward rose after reward, with post exceeding
> pre; Threat Inhibited and Danger Excited pre and post reward firing differed neither from zero
> nor from each other.

In [ ]:
def pre_post(tr, pops, rng):
    out = {}
    for p in nac.MAIN3:
        x = tr[pop_units(pops, [p])]
        x = x[np.all(np.isfinite(x), 1)]
        if len(x) < 3:
            for k in ("pre", "post", "post-pre"):
                out[f"{p[:2]} {k}"] = (np.nan, np.nan, np.nan)
            continue
        for k, v in (("pre", x[:, 7]), ("post", x[:, 8]), ("post-pre", x[:, 8] - x[:, 7])):
            m, lo, hi = nac.boot_ci(v, rng)
            out[f"{p[:2]} {k}"] = (round(m, 3), round(lo, 3), round(hi, 3))
    return out

rows = []
for key, pops in partitions.items():
    for k, (tr, _) in reward.items():
        if key != PRIMARY and k != ("a", "sd"):
            continue
        o = pre_post(tr, pops, np.random.default_rng(9101))
        if any(np.isnan(v[0]) for v in o.values()):
            rows.append(dict(partition=key, reading=k, **o, outcome="too few units"))
            continue
        bv = (not excl(o["Bi pre"]) and excl(o["Bi post"]) and o["Bi post"][0] > 0 and excl(o["Bi post-pre"])
              and o["Bi post-pre"][0] > 0 and abs(o["Bi post"][0] - 0.47) <= 0.15 and abs(o["Bi post-pre"][0] - 0.42) <= 0.16)
        others = not any(excl(o[f"{p} {w}"]) for p in ("Th", "Da") for w in ("pre", "post", "post-pre"))
        rows.append(dict(partition=key, reading=k, **o,
                         outcome="reproduced" if bv and others else ("partially_reproduced" if bv else "not_reproduced")))
t9b = pd.DataFrame(rows)
print(t9b.to_string(index=False))
print("paper: BV pre [-0.07, 0.13], post 0.47 [0.28, 0.62], post-pre 0.42 [0.21, 0.58]; TI and DE all CIs contain zero")
print("outcomes:", dict(collections.Counter(t9b.outcome)))

**Result: partially reproduced, under the script's reading and the assumed partition.**
Bidirectional Valence firing is near zero before reward (0.07 [-0.01, 0.17]) and above it after
(0.46 [0.31, 0.68]), with post above pre (0.39 [0.20, 0.61]), both within tolerance; Danger
Excited CIs all contain zero; but Threat Inhibited post-reward firing (0.24 [0.03, 0.49]) and its
rise (0.32 [0.04, 0.59]) exclude zero. Under ITI reading (b) the Bidirectional Valence part
fails: its pre-reward CI excludes zero with normalization (i), and its post-reward mean (0.88)
is outside the tolerance with normalization (ii).

## fig9efg-reward-cue-correlations

> Bidirectional Valence reward onset firing correlated positively with safety onset firing and
> negatively with danger firing, and the two correlations differed; ...

Reward onset firing is the post bin of the reward trace; safety onset is the first 1 s and
danger the whole 10-s cue of the normalized cue traces (N1, the authors' `cue0onset` and
`cue100entire`). Comparison readings: (a) Fisher r-to-z for independent correlations; (b)
Steiger's test for dependent correlations sharing a variable.

In [ ]:
cue_on_s = norm[PRIMARY_NORM][:, 2, nac.ONSET].mean(1)
cue_whole_d = norm[PRIMARY_NORM][:, 0, nac.WHOLE_CUE].mean(1)

def reward_corr(tr, pops):
    out = {}
    for p in nac.MAIN3:
        idx = np.array(pop_units(pops, [p]))
        ok = np.isfinite(tr[idx, 8])
        rw, s, d = tr[idx, 8][ok], cue_on_s[idx][ok], cue_whole_d[idx][ok]
        n = len(rw)
        if n < 4:
            out[p] = dict(n=n, r_safety=np.nan, R2_safety=np.nan, p_safety=np.nan, r_danger=np.nan, R2_danger=np.nan,
                          p_danger=np.nan, fisher_p=np.nan, steiger_p=np.nan)
            continue
        rs, rd, rsd = stats.pearsonr(rw, s), stats.pearsonr(rw, d), stats.pearsonr(s, d).statistic
        out[p] = dict(n=n, r_safety=rs.statistic, R2_safety=rs.statistic**2, p_safety=rs.pvalue, r_danger=rd.statistic,
                      R2_danger=rd.statistic**2, p_danger=rd.pvalue,
                      fisher_p=nac.fisher_independent(rs.statistic, n, rd.statistic, n)[1],
                      steiger_p=nac.steiger_dependent(rs.statistic, rd.statistic, rsd, n)[1])
    return out

rows = []
for key, pops in partitions.items():
    for k, (tr, _) in reward.items():
        if key != PRIMARY and k != ("a", "sd"):
            continue
        o = reward_corr(tr, pops)
        if any(o[p]["n"] < 4 for p in nac.MAIN3):
            for p in nac.MAIN3:
                rows.append(dict(partition=key, reading=k, pop=p, **o[p], outcome="too few units"))
            continue
        res = []
        for cmp in ("fisher_p", "steiger_p"):
            b, ti, de = o["Bidirectional Valence"], o["Threat Inhibited"], o["Danger Excited"]
            bv = b["r_safety"] > 0 and b["p_safety"] < 0.05 and b["r_danger"] < 0 and b["p_danger"] < 0.05 and b[cmp] < 0.05
            ti_ok = ti[cmp] >= 0.05
            rest = ti["p_safety"] >= 0.05 and de["p_safety"] >= 0.05 and de["p_danger"] >= 0.05 and de[cmp] < 0.05
            res.append("reproduced" if bv and ti_ok and rest else ("partially_reproduced" if bv and ti_ok or bv else "not_reproduced"))
        best = "reproduced" if "reproduced" in res else ("partially_reproduced" if "partially_reproduced" in res else "not_reproduced")
        for p in nac.MAIN3:
            rows.append(dict(partition=key, reading=k, pop=p, **o[p], outcome=best))
t9e = pd.DataFrame(rows)
print(t9e.round(4).to_string(index=False))
print("paper: BV R2 0.08 (p = 0.006) safety, 0.12 (p = 8.4e-4) danger, Z = -3.96; TI 0.07 (p = 0.18), 0.15 (p = 0.05), Z = -0.50; DE 0.06 (p = 0.07), 0.06 (p = 0.07), Z = -2.10")

**Result: partially reproduced, resting on the assumed partition.** Under ITI reading (b) with
either normalization and under the script's reading, Bidirectional Valence reward onset firing
correlates positively with safety onset firing (script reading R2 = 0.088, p = 0.005) and
negatively with danger firing (R2 = 0.109, p = 0.002), the two correlations differ (Fisher and
Steiger p of 0.002 or less), and Threat Inhibited shows no difference between its correlations (p 0.20 to
0.49). The Danger Excited pattern does not hold: its danger correlation is significant (p 0.02 to
0.03). Under the script's reading the R2 values (0.088 and 0.109) are close to the paper's 0.08
and 0.12; the first is just outside its tolerance of 0.008.

Note on the criterion's partial outcome: the claims file's text reads "partially reproduced when
the Bidirectional Valence part holds and the Threat Inhibited or Danger Excited pattern of the
statement does not". The table's outcome column gives "reproduced" only when the Bidirectional
Valence part, the Threat Inhibited test of no difference, and the rest of the pattern all hold.

## fig9hij-danger-safety-correlations

> Danger firing correlated negatively with safety onset firing in Danger Excited and
> Bidirectional Valence units but not in Threat Inhibited units.

In [ ]:
rows = []
for key, pops in partitions.items():
    for nr in (list(nac.NORM_READINGS) + ["script normalization"] if key == PRIMARY else [PRIMARY_NORM]):
        s_on = norm[nr][:, 2, nac.ONSET].mean(1)
        d_wh = norm[nr][:, 0, nac.WHOLE_CUE].mean(1)
        row = dict(partition=key, normalization=nr)
        for p in nac.MAIN3:
            idx = pop_units(pops, [p])
            r = stats.pearsonr(d_wh[idx], s_on[idx])
            row[f"{p[:2]} n"] = len(idx)
            row[f"{p[:2]} r"] = r.statistic
            row[f"{p[:2]} R2"] = r.statistic ** 2
            row[f"{p[:2]} p"] = r.pvalue
        ok = row["Da r"] < 0 and row["Da p"] < 0.05 and row["Bi r"] < 0 and row["Bi p"] < 0.05 and row["Th p"] >= 0.05
        row["outcome"] = "reproduced" if ok else "not_reproduced"
        rows.append(row)
t9h = pd.DataFrame(rows)
print(t9h.round(4).to_string(index=False))
print("paper: TI R2 0.06, p = 0.24; DE R2 0.25 (+-0.025), p = 1.2e-4; BV R2 0.06 (+-0.006), p = 0.015")
print("outcomes:", dict(collections.Counter(t9h.outcome)))

**Result: not reproduced, resting on the assumed partition.** With the claims file's
normalization, Danger Excited danger firing correlates negatively with safety onset firing (R2
0.11 to 0.14, p 0.01 to 0.03, below the paper's 0.25), but Bidirectional Valence firing does not
(R2 0.008 to 0.011, p 0.33 to 0.40); Threat Inhibited shows no correlation (p 0.33 to 0.64). With
the script normalization (variant) the pattern holds (DE R2 0.21, p = 0.001; BV R2 0.062,
p = 0.019; TI p = 0.44).

## fig10a-poke-cessation-anova

> The three populations showed minimal firing changes around nose poke cessation in the ITI;
> there was no main effect of interval, and a population by interval interaction driven by
> modest Bidirectional Valence increases.

In [ ]:
rows = []
for key, pops in partitions.items():
    for k, (tr, _) in cess.items():
        if key != PRIMARY and k != ("a", "sd"):
            continue
        A, per, groups = event_anova(tr, pops)
        ok = A["level"][3] >= 0.05 and A["group x level"][3] < 0.05
        rows.append(dict(partition=key, reading=k, n=sum(len(g) for g in groups),
                         F_bin=A["level"][0], df_bin=f"{A['level'][1]},{A['level'][2]}", p_bin=A["level"][3],
                         F_x=A["group x level"][0], df_x=f"{A['group x level'][1]},{A['group x level'][2]}", p_x=A["group x level"][3],
                         outcome="reproduced" if ok else "not_reproduced"))
t10a = pd.DataFrame(rows)
print(t10a.round(4).to_string(index=False))
print("paper: interval F(15,2535) = 1.58, p = 0.071; interaction F(30,2535) = 2.18, p = 2.35e-4")
print("outcomes:", dict(collections.Counter(t10a.outcome)))

**Result: reproduced, under the script's reading and the assumed partition.** With the script's
ITI window and normalization there is no effect of interval (F(15,2385) = 0.96, p = 0.49) and a
population by interval interaction (F(30,2385) = 1.70, p = 0.010). Under ITI reading (b) the
interval effect is significant (p < 1e-4), so the outcome rests on the script's reading.

## fig10bcd-poke-cessation-pre-post

> Pre and post nose poke cessation firing differed neither from zero nor from each other in
> Threat Inhibited and Danger Excited units; in Bidirectional Valence units both exceeded zero
> but did not differ.

In [ ]:
rows = []
for key, pops in partitions.items():
    for k, (tr, _) in cess.items():
        if key != PRIMARY and k != ("a", "sd"):
            continue
        o = pre_post(tr, pops, np.random.default_rng(10101))
        if any(np.isnan(v[0]) for v in o.values()):
            rows.append(dict(partition=key, reading=k, **o, outcome="too few units"))
            continue
        others = not any(excl(o[f"{p} {w}"]) for p in ("Th", "Da") for w in ("pre", "post", "post-pre"))
        bv = excl(o["Bi pre"]) and o["Bi pre"][0] > 0 and excl(o["Bi post"]) and o["Bi post"][0] > 0 and not excl(o["Bi post-pre"])
        rows.append(dict(partition=key, reading=k, **o, outcome="reproduced" if others and bv else "not_reproduced"))
t10b = pd.DataFrame(rows)
print(t10b.to_string(index=False))
print("outcomes:", dict(collections.Counter(t10b.outcome)))

**Result: reproduced, under ITI reading (b) and the assumed partition.** Threat Inhibited and
Danger Excited pre, post and difference CIs contain zero; Bidirectional Valence pre (0.24 [0.08,
0.43]) and post (0.35 [0.16, 0.55]) exceed zero and do not differ (0.11 [-0.04, 0.23]), under
both normalizations of reading (b). Under the script's reading the Danger Excited pre CI excludes
zero below it and the Bidirectional Valence pre CI contains zero.

## fig10efg-poke-cessation-cue-unrelated

> Nose poke cessation firing was unrelated to safety onset and danger cue firing in all three
> populations ...

In [ ]:
rows = []
for key, pops in partitions.items():
    for k, (tr, _) in cess.items():
        if key != PRIMARY and k != ("a", "sd"):
            continue
        o = reward_corr(tr, pops)
        if any(o[p]["n"] < 4 for p in nac.MAIN3):
            for p in nac.MAIN3:
                rows.append(dict(partition=key, reading=k, pop=p, n=o[p]["n"], outcome="too few units"))
            continue
        ps = [o[p][c] for p in nac.MAIN3 for c in ("p_safety", "p_danger")]
        for p in nac.MAIN3:
            rows.append(dict(partition=key, reading=k, pop=p, n=o[p]["n"], R2_safety=o[p]["R2_safety"], p_safety=o[p]["p_safety"],
                             R2_danger=o[p]["R2_danger"], p_danger=o[p]["p_danger"],
                             outcome="reproduced" if min(ps) >= 0.05 else "not_reproduced"))
t10e = pd.DataFrame(rows)
print(t10e.round(4).to_string(index=False))
print("paper: all R2 < 0.02, p > 0.45")

**Result: not reproduced, resting on the assumed partition.** Under ITI reading (b) and under the
script's reading, Danger Excited cessation firing correlates with danger cue firing (R2 0.19 to
0.24, p 0.0006 to 0.003); the other five correlations have p > 0.16. Reading (a), with at most
four cessation events per unit, finds no correlation.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| units-recorded | partially reproduced: 368 units and 7 rats, but 90 sessions, not 95 |
| cue-responsive-193 | reproduced (211 units under reading b; the script's procedure gives 193 exactly) |
| fig1c-acquisition | blocked: data missing |
| fig1g-discrimination | reproduced under one reading, with a CI bound near zero |
| fig1h-poke-rate-unrelated | reproduced (assumed 0/0 handling) |
| fig1h-discrimination-safety | reproduced |
| fig2-five-clusters | partially reproduced: the patterns, not the counts |
| populations-by-rat | not reproduced (assumed partition) |
| most-units-threat | reproduced |
| fig3ab-rate-cv-unimodal | reproduced |
| fig3c-waveform-bimodal | reproduced (assumed partition) |
| fig4ab-thi-threat-firing | partially reproduced (assumed partition) |
| fig4b-thi-danger-equals-uncertainty | reproduced (assumed partition) |
| fig4cd-de-danger-firing | partially reproduced (assumed partition) |
| fig4ef-bv-danger-decrease | partially reproduced (assumed partition) |
| fig4f-bv-safety-variability | reproduced (assumed partition) |
| fig5ab-the-onset | not reproduced (assumed partition) |
| fig5cd-ns-nonselective | not reproduced (assumed partition) |
| fig6-similarity-anova | reproduced (assumed partition) |
| fig6d-thi-similarity | reproduced (assumed partition) |
| fig6e-de-similarity | not reproduced (assumed partition) |
| fig6f-bv-no-similarity-difference | reproduced (assumed partition) |
| fig6-thi-greatest-similarity | reproduced (assumed partition) |
| fig7-tuning-curves | reproduced (assumed partition) |
| fig8ab-thi-exaggerated-threat | reproduced under one reading (assumed partition) |
| fig8cd-de-exaggerated-threat | partially reproduced (assumed partition) |
| fig8ef-bv-exaggerated-threat | partially reproduced (assumed partition) |
| fig9a-reward-firing | not reproduced (assumed partition) |
| fig9bcd-reward-pre-post | partially reproduced (assumed partition and script reading) |
| fig9efg-reward-cue-correlations | partially reproduced (assumed partition) |
| fig9hij-danger-safety-correlations | not reproduced (assumed partition) |
| fig10a-poke-cessation-anova | reproduced (assumed partition and script reading) |
| fig10bcd-poke-cessation-pre-post | reproduced (assumed partition) |
| fig10efg-poke-cessation-cue-unrelated | not reproduced (assumed partition) |

**How the analysis differs from the paper's.** The populations are not the paper's: its labels
are not deposited, and its k-means cannot be repeated (one MATLAB start after `rng(1)`), so the
population claims rest on a reproduced partition chosen among the claims file's readings. The
cue firing normalization follows the Methods; the authors' script normalizes differently (per
trial, minus a floored 2-s baseline, z-scored over all trials), and that variant brings several
effect sizes to the paper's values (fig4ab, fig4cd, fig8ef, fig9hij). The cue-responsive set is
the script's (floored rates), the only procedure that gives 193 units. The ITI events follow the
script's windows where the claims file's first reading cannot apply. The bootstrap intervals
are standard studentized intervals; the paper's printed intervals match a form that does not
reflect the bootstrap-t quantiles.

**What was found about the data set.** It holds all 368 units and 7 rats, from 90 sessions
where the paper reports 95. The pre-surgery sessions and the paper's population labels are not
in it. The data description does not say which event channel is which (the script does), nor
that the recording is paused between trials, so that the files hold only 20 s before to about
33 s after each onset; it describes `poke.base` as a 2-s rate where the values are 20-s rates.
The script's waveform measure is half the scale of Figure 3C.